<div style="background-color: #cbe1f5ff; padding: 15px; border-radius: 8px;">
  <h1>Estimación de la Demanda de Consumo en EE.UU en 1987T4 – 2019T4/2024T2 </h1>

  El objetivo del estudio es estimar una función de consumo trimestral para Estados Unidos, utilizando datos de la base FRED (Federal Reserve Bank of St. Louis).  
  Se plantea un modelo dinámico con variables macroeconómicas clave y dummies para capturar los efectos del Covid-19 2020T1 – 2021T2.

  La especificación general es:
$$
\Delta \ln(c_t) = \beta_0 + \beta_1 \left[\ln(y_{t}^d) - \ln(c_{t-1})\right] + \beta_2 r_t^{cp} + \beta_3 r_t^{lp} + \beta_4 \Delta u_t + \beta_5 (\text{gap}_{y}^d)t + \beta_6 \frac{AL_{t-1}}{y_t^d} + \beta_7 \frac{IA_{t-1}}{y_t^d} + \epsilon_t
$$


  ---

  ## Variables del modelo

  •⁠  ⁠$c_t$: Consumo privado real  
    Medido en miles de millones de USD, ajustado por inflación.  
    Fuente: Personal Consumption Expenditures (PCE), FRED.  
    Se utiliza la variación logarítmica $\Delta ln(C_t)$ para aproximar la tasa de crecimiento.

  •⁠  ⁠$y_t^d$: Renta disponible  
    Variable fundamental que explica la capacidad de gasto de los hogares.  
    Fuente: Real Disposable Personal Income, FRED.  
    Se introduce en logaritmos.

  •⁠  ⁠$c_{t-1\,}$: Consumo del periodo anterior 
    Captura la inercia y persistencia del consumo en el tiempo.  
    Permite modelar el carácter dinámico del gasto de los hogares.

  •⁠  ⁠$r_t^{cp}$, $r_t^{lp}$: Tipos de interés real de corto y largo plazo
    Fuente: 1-Year Real Interest Rate and 10-Year Real Interest Rate, FRED.
    Dado que ambos tipos de interés están registrados en frecuencia mensual, se ajustan los tipos de interés a frecuencia trimestral a partir de los promedios trimestrales.

  •⁠  ⁠$\Delta u_t$: Variación en la tasa de desempleo
    Fuente: Unemployment Rate, FRED.
    Estos datos también vienen dados en frecuencia mensual, por lo que se ajustan de la misma forma que los tipos de interés.
    Se incluye para estudiar cómo la variación en la tasa de paro afecta al consumo , como posible indicadora de la incertidumbre económica.
  
  •⁠  ⁠$\text{gap}_{y}^d$: Diferencia entre la renta disponible y el promedio de los últimos 4 periodos
    Se incluye para capturar el efecto de los cambios de tendencia en la renta disponible al consumo.

  •⁠  ⁠$AL_{t-1\,}$: Medida de los activos líquidos.
    Para ello se emplea M3. Como viene medido en términos nominales y en millones de dólares, mientras que el consumo viene dado en billones de dóleres, se procede a dividir entre $10^9$ y entre el IPC de US (Fuente: Consumer Price Index for All Urban Consumers)

  •⁠ $IA_{t-1}$: Medida de los activos ilíquidos.
  Representa la riqueza financiera o patrimonial menos líquida de los hogares, es decir, aquella que no puede convertirse rápidamente en dinero sin pérdida significativa de valor.  
  En este contexto, los activos ilíquidos incluyen viviendas, acciones, bonos a largo plazo, fondos de pensiones y otros instrumentos de inversión no monetarios.  
  Se introducen en el modelo para capturar el efecto riqueza: cuando los precios de los activos ilíquidos aumentan (por ejemplo, el valor inmobiliario o bursátil), los hogares tienden a incrementar su consumo, aun cuando su renta corriente no cambie.  
  Sin embargo, el impacto de estos activos sobre el consumo suele ser menor y más lento que el de los activos líquidos, ya que su conversión en gasto requiere decisiones de desinversión o financiación.  
  En la especificación, \( \frac{IA_{t-1}}{y_t^d} \) mide la proporción de riqueza ilíquida respecto a la renta disponible, indicando el grado en que la posición patrimonial de los hogares puede sostener el consumo ante shocks de ingresos.


  •⁠  ⁠$DUM_{Covid\,\,\,}$: Variables dummy (2020_T1 – 2021_T2)
    Seis dummies trimestrales que toman valor 1 en el trimestre correspondiente y 0 en los demás.  
    Capturan el efecto extraordinario de la pandemia sobre el consumo.

  •⁠  ⁠$ε_t$: Término de error  
    Recoge factores no recogidos en las variables macroeconómicas empleadas que afectan al consumo.

  ---

  ## Concepto general del estudio

  El análisis busca comprender los determinantes del consumo privado en EE.UU. desde finales de los 80 hasta la actualidad, incorporando:

  •⁠  ⁠Efecto renta disponible (determinante clave del consumo).  
  •⁠  ⁠Persistencia e inercia del consumo pasado.  
  •⁠  ⁠Efecto riqueza y liquidez con factores financieros que influyen en las decisiones de gasto.  
  •⁠  ⁠Impacto del Covid-19 a través de shocks temporales capturados con dummies.  

  Se trata de un modelo de corte macroeconómico y econométrico dinámico, que permitirá estimar elasticidades de consumo y analizar cómo distintos factores explican su evolución en el largo y corto plazo.
</div>


## Importación de Librerías


In [90]:
# ---- 0) Imports
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.stats.api as sms
from statsmodels.stats.diagnostic import het_white, acorr_ljungbox
from statsmodels.tsa.stattools import adfuller
import openpyxl

## Tratamiento de los Datos

In [91]:
# Cargar los datos desde el el Excel
file_path = 'data_c_eeuu.xlsx'  # Ajusta la ruta
df = pd.read_excel(file_path) 
df = df.dropna().copy()
df.head()
df.isnull().sum()


t              0
delta_ln_ct    0
y_t^d          0
r_t^cp         0
r_t^lp         0
delta_u_t      0
gap_y^d        0
AL_t-1         0
IA_t-1         0
ln_yt          0
ln_ct-1        0
dtype: int64

In [92]:
# ---- 2) Convertir las fechas a formato datetime
df['t'] = pd.to_datetime(df['t'], format='%Y-%m-%d')

# ---- 4) Crear dummies Covid (2020T1 - 2021T2)
# Considera las fechas específicas de los trimestres 2020T1 a 2021T2 para crear las dummies

covid_periods = [
    '2020-01-01', '2020-04-01', '2020-07-01', '2020-10-01',
    '2021-01-01', '2021-04-01', '2021-07-01'
]

df['AL/y'] = df['AL_t-1'] / df['y_t^d']  # Tasa de desempleo
df['IA/y'] = df['IA_t-1'] / df['y_t^d']  # Inversión bruta en bienes de equipo sobre PIB
df['y-c'] = df['ln_yt'] - df['ln_ct-1']  # Ahorro

# Asignar las dummies
for date in covid_periods:
    df[date] = np.where(df['t'] == date, 1, 0)

## Construcción del Modelo

In [93]:
# Regresión propuesta con todas las variables independientes
X = df[['y-c', 'r_t^cp', 'r_t^lp', 'delta_u_t', 'gap_y^d', 'AL/y', 'IA/y'] + covid_periods]  # Variables independientes
y = df['delta_ln_ct']  # Variable dependiente

# Añadir constante para el modelo (intercepto)
X = sm.add_constant(X)

#6) Estimación del modelo (OLS)
model = sm.OLS(y, X, missing='drop')  # Estimación por OLS 
results = model.fit()
print(results.summary())
# Para verlo en Latex 
with open("resultados_regresion.tex", "w") as f:
    f.write(results.summary().as_latex())

                            OLS Regression Results                            
Dep. Variable:            delta_ln_ct   R-squared:                       0.861
Model:                            OLS   Adj. R-squared:                  0.847
Method:                 Least Squares   F-statistic:                     59.83
Date:                Mon, 06 Oct 2025   Prob (F-statistic):           9.52e-51
Time:                        20:44:55   Log-Likelihood:                 598.59
No. Observations:                 150   AIC:                            -1167.
Df Residuals:                     135   BIC:                            -1122.
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1563      0.089      1.758      0.0

In [94]:
# Test de heterocedasticidad (White)
white_test = het_white(results.resid, results.model.exog)
print("White Test for Heteroscedasticity:")
print(white_test)

White Test for Heteroscedasticity:
(57.72878437527887, 0.03440305532127673, 1.7048755276232477, 0.01579079633206748)


El Test de White permite hacer el contraste de las siguiente hipótesis:

$H_0$: Los errores poseen homocedasticidad.

$H_1$: Los errors poseen heterocedasticidad.

Como los p-valores del test de white son menores de 0.05 podemos rechazar la hipotesis nula, afirmando que existe heterocedasticidad en el modelo. 

In [95]:
# Test de autocorrelación 
from statsmodels.stats.stattools import durbin_watson
dw = durbin_watson(results.resid)
print("Durbin-Watson:", dw)


Durbin-Watson: 1.8995382561397893


In [96]:
from statsmodels.stats.diagnostic import acorr_breusch_godfrey
bg_test = acorr_breusch_godfrey(results, nlags=4)  
print("Breusch-Godfrey Test:", bg_test)


Breusch-Godfrey Test: (6.293259810311774, 0.17829175956465468, 1.434200361901327, 0.22624617041299117)


De igual manera se realizó tanto el test de Durbin-Watson para mirar autocorrelación de primer orden como el de Breusch-Godfrey que es de carácter más general. 

Tras ninguno de los dos podemos rechazar la hipótesis nula. Es decir no podemos afirmar que exista autocorrelación en los errores.

## Modelo con Errores HAC 

In [97]:
results_hac = results.get_robustcov_results(cov_type='HAC', maxlags=4)
print(results_hac.summary())




                            OLS Regression Results                            
Dep. Variable:            delta_ln_ct   R-squared:                       0.861
Model:                            OLS   Adj. R-squared:                  0.847
Method:                 Least Squares   F-statistic:                 1.466e+04
Date:                Mon, 06 Oct 2025   Prob (F-statistic):          2.19e-194
Time:                        20:44:55   Log-Likelihood:                 598.59
No. Observations:                 150   AIC:                            -1167.
Df Residuals:                     135   BIC:                            -1122.
Df Model:                          14                                         
Covariance Type:                  HAC                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1563      0.089      1.749      0.0

/Users/miguelfernandez/Desktop/Programación/Universidad/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 14, but rank is 8
  warnings.warn('covariance of constraints does not have full '


## MODELO CON VARIABLES SIGNIFICATIVAS


In [98]:
# Crear un DataFrame con los coeficientes y los valores p
params = results.params
p_values = results.pvalues

# Filtrar los coeficientes significativos (valor p < 0.05)
significant_params = params[p_values < 0.05]
print(significant_params)


delta_u_t    -0.007537
gap_y^d       0.000011
2020-01-01   -0.021455
2020-04-01   -0.036253
2020-07-01    0.049267
2021-04-01    0.027314
dtype: float64


In [99]:
X = significant_params.index.tolist()
U = df['delta_ln_ct']  # Variable dependiente
X = df[X]  # Variables independientes
X = sm.add_constant(X)
model = sm.OLS(U, X, missing='drop')  # Estimación por OLS
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 4})
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:            delta_ln_ct   R-squared:                       0.840
Model:                            OLS   Adj. R-squared:                  0.833
Method:                 Least Squares   F-statistic:                     111.3
Date:                Mon, 06 Oct 2025   Prob (F-statistic):           3.13e-37
Time:                        20:44:55   Log-Likelihood:                 587.81
No. Observations:                 150   AIC:                            -1162.
Df Residuals:                     143   BIC:                            -1141.
Df Model:                           6                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0055      0.001      9.208      0.0

/Users/miguelfernandez/Desktop/Programación/Universidad/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 6, but rank is 3
  warnings.warn('covariance of constraints does not have full '


<div style="background-color: #ff7e7ef0; padding: 15px; border-radius: 8px;">

  <h2>C1. Discusión e interpretación de la especificación propuesta</h2>

  La especificación empírica propuesta:

  $$
  \Delta \ln(c_t) = \beta_0 + \beta_1 \left[\ln(y_{t}^d) - \ln(c_{t-1})\right] + \beta_2 r_t^{cp} + \beta_3 r_t^{lp} + \beta_4 \Delta u_t + \beta_5 (\text{gap}_{y}^d)t + \beta_6 \frac{AL_{t-1}}{y_t^d} + \beta_7 \frac{IA_{t-1}}{y_t^d} + \epsilon_t
  $$

  puede interpretarse como una **forma versión extensiva** derivada de los modelos de **elección intertemporal de consumo y ocio** vistos en clase, cuya formulación básica es:

  $$
  \ln(c_1) + \gamma \ln(1-n_1) + \beta[\ln(c_2) + \gamma \ln(1-n_2)]
  $$

  donde el consumidor elige consumo $(c_t)$ y ocio $(1-n_t)$ maximizando utilidad intertemporal sujeta a una restricción presupuestaria que vincula los ingresos presentes y futuros.  

  En este marco teórico, la decisión de consumo depende de tres grandes dimensiones:  
  1. **Expectativas de renta futura** (ingreso permanente o renta disponible esperada).  
  2. **Preferencias por consumo presente vs. futuro**, capturadas por el tipo de interés real y el parámetro de descuento β.  
  3. **Condiciones del mercado laboral**, que afectan tanto a la renta como al ocio disponible.  

  ---

  ### Correspondencia entre la teoría y la especificación empírica

  - **Término dinámico $[\ln(y_t^d) - \ln(c_{t-1})]$**  
    Representa la **brecha entre la renta disponible actual y el consumo pasado**.  
    En los modelos intertemporales, los consumidores suavizan el consumo en el tiempo, ajustándolo lentamente ante cambios en la renta.  
    Por tanto, este término actúa como una **condición de equilibrio de largo plazo**: si la renta permanente aumenta más rápido que el consumo, el término será positivo y la variación del consumo ($\Delta \ln c_t$) tenderá a ser mayor.

  - **Términos de tipo de interés $r_t^{cp}$ y $r_t^{lp}$**  
    En la teoría consumo-ocio, el tipo de interés real refleja el **precio relativo del consumo presente respecto al futuro**.  
    Un aumento de $r_t$ encarece el consumo actual y, teóricamente, incentiva el ahorro.  
    Su efecto empírico suele ser pequeño o ambiguo, ya que depende de la magnitud relativa de los efectos **sustitución** (consumir menos hoy) y **renta** (mayor ingreso financiero futuro).  

  - **Variación del desempleo ($\Delta u_t$)**  
    Captura el componente del modelo teórico relacionado con el **riesgo de ingresos y disponibilidad de ocio**.  
    Cuando el desempleo aumenta, se reduce la renta esperada y la confianza del consumidor, generando una caída en el consumo presente.  
    En términos del modelo teórico, este efecto se traduce en un desplazamiento de la restricción presupuestaria y un aumento involuntario del “ocio”.

  - **Diferencia entre la renta disponible y su media reciente $(\text{gap}_{y}^d)$**  
    Este término aproxima la **variación transitoria de la renta respecto a su tendencia**.  
    En los modelos de consumo permanente, los consumidores responden más a los cambios permanentes que a los transitorios.  
    Por tanto, un $\beta_5$ positivo indicaría que los hogares interpretan el “gap” como un cambio más duradero en su ingreso, y ajustan el consumo al alza.

  - **Activos líquidos e ilíquidos $(AL_{t-1}/y_t^d,\ IA_{t-1}/y_t^d)$**  
    Estos términos introducen una dimensión de **restricciones de liquidez** que la teoría básica no incluye explícitamente, pero que son fundamentales empíricamente.  
    Si los hogares enfrentan limitaciones de crédito, la disponibilidad de activos líquidos (M3 ajustado por IPC) condiciona su capacidad de suavizar el consumo ante shocks de renta.

  - **Dummies del Covid-19**  
    Actúan como **shocks exógenos de corto plazo** que alteran simultáneamente la renta, la capacidad de gasto y las expectativas.  
    Desde el punto de vista teórico, representan desplazamientos temporales de la restricción presupuestaria y cambios abruptos en la utilidad derivada del ocio (por confinamientos o cambios en el tiempo de trabajo).

  ---

  ### Interpretación global

  El modelo empírico combina los fundamentos microeconómicos del modelo de **consumo-ocio intertemporal** con extensiones macroeconómicas y financieras:

  - La **presencia de un término de corrección dinámica** aproxima la idea de que los consumidores ajustan gradualmente el consumo hacia un nivel consistente con la renta permanente.  
  - Los **tipos de interés reales** recogen la intertemporalidad de la decisión: consumir hoy o ahorrar para el futuro.  
  - Las **variables laborales y financieras** introducen la incertidumbre y las restricciones de liquidez reales que limitan el modelo puramente teórico.  
  - Finalmente, las **dummies de la pandemia** permiten capturar perturbaciones excepcionales no explicadas por los fundamentos económicos.

  ---

  En síntesis, la especificación propuesta puede entenderse como una **versión empírica ampliada del modelo de elección intertemporal de consumo-ocio**, en la que se reconoce que las decisiones de los hogares no solo dependen de preferencias y precios relativos entre consumo presente y futuro, sino también de factores de incertidumbre, liquidez y shocks macroeconómicos de corto plazo.

</div>


<div style="background-color: #ffc0a8f0; padding: 15px; border-radius: 8px;">
  <h2>C2. Análisis del Modelo de Demanda de Consumo en EE.UU.</h2>

  ## Resumen del Modelo

  El modelo propuesto para estimar la función de consumo privado en EE.UU. considera diversas variables macroeconómicas clave que afectan al consumo. Se incluyeron variables como la renta disponible, los tipos de interés, el desempleo, la diferencia entre la renta disponible y su media, y la medida de activos líquidos e ilíquidos, además de variables dummy para capturar el impacto del Covid-19. La regresión se ajustó mediante Mínimos Cuadrados Ordinarios (OLS), con un total de 150 observaciones y un período de análisis que abarca de 1987_T4 a 2019_T4, con un enfoque adicional en los efectos de la pandemia de Covid-19 entre 2020_T1 y 2021_T2.

  ### Resultados del Modelo Completo

  Los resultados iniciales del modelo proporcionan una visión global del ajuste y la significancia de las variables:

  - **R-cuadrado**: El valor de 0.861 indica que el modelo explica el 86.1% de la variabilidad en la variable dependiente, la variación del consumo privado en logaritmos. Este es un valor relativamente alto, lo que sugiere que el modelo tiene un buen poder explicativo. Es usual en modelo macroeconómicos tener estas cifras.
  
  - **Significancia de los coeficientes**: Se observan varias variables no significativas (como los tipos de interés real de corto y largo plazo, y los activos ilíquidos), lo que indica que estos factores no tienen un efecto relevante en la explicación de las fluctuaciones en el consumo.

  - **Variables con significancia**: 
    - **Variación en la tasa de desempleo $(Δu_t)$**: La variable es estadísticamente significativa con un valor p de 0.001, lo que sugiere que las fluctuaciones en el desempleo tienen un impacto importante en el consumo privado.
    - **Gap de renta disponible $(gap_y^d)$**: El coeficiente es significativo a un nivel del 1% (p=0.007), lo que indica que las diferencias en la renta disponible respecto al promedio de los últimos 4 trimestres tienen un efecto sobre el consumo.
    - **Dummies del Covid-19**: Las variables dummy correspondientes a los trimestres de $2020$ y $2021$ son altamente significativas y muestran un efecto importante en el consumo. El impacto negativo en $2020$, seguido de una recuperación a mitad de $2021$, es un reflejo de los efectos económicos de la pandemia.

  ---

  ### Resultados del Modelo Reducido (Eliminando Variables No Significativas)

  En el análisis posterior, se eliminaron las variables no significativas del modelo, lo que resulta en un modelo reducido con 6 variables significativas. Los resultados son los siguientes:

  - **R-cuadrado**: El valor ajustado de $0.833$ y un R-cuadrado de $0.840$ indican que, aunque el modelo es un poco menos explicativo que el modelo completo, sigue siendo robusto para capturar la dinámica del consumo en EE.UU. en el largo plazo.
  
  - **Coeficientes significativos**: Las variables **$Δu_t$**, **$gap_y^d$**, y las **dummies del Covid-19** siguen siendo significativas en este modelo reducido, con **$Δu_t$** teniendo un impacto negativo en el consumo y **$gap_y^d$** mostrando una relación positiva. Además, las dummies de la pandemia continúan reflejando un impacto significativo.

    - **Dummys de Covid-19**: Los coeficientes de las dummies continúan siendo altamente significativos, especialmente para los trimestres de $2020$ y $2021$. El efecto negativo en el primer trimestre de $2020$ ($-0.0239$) y en abril de $2020$ ($-0.0480$) refleja la caída del consumo durante los picos iniciales de la pandemia. Sin embargo, a partir del tercer trimestre de $2020$, se observa una recuperación en el consumo ($+0.0486 $en julio de $2020$), lo que podría reflejar los intentos de reactivación económica.

  ---

  ### Discusión de los Resultados

  1. **Efecto de la renta disponible**: El **$gap_y^d$** muestra un efecto positivo en el consumo, lo que confirma la teoría económica básica de que los hogares aumentan su consumo cuando la renta disponible está por encima de su tendencia reciente. Este efecto es consistente con los modelos de consumo intertemporal, donde los hogares ajustan su gasto en función de cambios en su renta disponible.

  2. **Impacto de la tasa de desempleo**: La variable **$Δu_t$** tiene un coeficiente negativo significativo, lo que sugiere que un aumento en la tasa de desempleo reduce el consumo privado. Esto es consistente con la teoría económica, que indica que un aumento en el desempleo genera incertidumbre y reduce la confianza de los consumidores, afectando negativamente su disposición a gastar.

  3. **Efectos de la pandemia (Covid-19)**: Las dummies de Covid-19 proporcionan una clara indicación del impacto económico de la pandemia. Los resultados indican una fuerte caída en el consumo al principio de la crisis, con una posterior recuperación en los trimestres siguientes. Este hallazgo es relevante para comprender el impacto transitorio de eventos extraordinarios, como la pandemia, en las decisiones de consumo de los hogares.

  ---

  ### Consideraciones

  - **Persistencia del consumo**: Aunque el modelo original incluía el consumo del período anterior ($c_{t-1}$), que es una variable relevante para capturar la persistencia del consumo, esta variable no mostró significancia en el modelo completo ni en el reducido. Esto podría sugerir que el consumo no es completamente dependiente del consumo pasado en este caso, o que otras variables macroeconómicas juegan un rol más destacado en la explicación de las fluctuaciones en el consumo.

  - **Tipos de interés**: A pesar de la teoría económica que sugiere que los tipos de interés pueden influir en el consumo a través del coste del crédito, los resultados muestran que los tipos de interés real de corto y largo plazo no son significativos. Esto podría ser indicativo de que otros factores, como la renta disponible y la variabilidad del desempleo, son más relevantes para explicar las decisiones de consumo en el contexto estadounidense.

</div>


